# 02 - Transformer Baseline + MIA Attacks

Ce notebook extrait la partie baseline Transformer et les 3 attaques MIA
du notebook original: threshold_loss, logistic, shadow_meta.

In [1]:
import os
import random
import numpy as np
import pandas as pd

os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.metrics import roc_curve

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
def set_global_determinism(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def build_transformer(n_features, d_model=64, n_heads=4, ff_dim=128, dropout=0.0):
    inp = layers.Input(shape=(1, n_features))

    x = layers.Dense(d_model)(inp)
    attn = layers.MultiHeadAttention(num_heads=n_heads, key_dim=max(1, d_model // n_heads), dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff)

    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def mia_features(proba, y_true):
    eps = 1e-8
    p = np.clip(proba, eps, 1.0 - eps)
    loss = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    conf = np.maximum(p, 1 - p)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    cor = ((p >= 0.5).astype(int) == y_true).astype(float)
    margin = np.abs(p - 0.5)
    return np.column_stack([loss, conf, ent, cor, margin])


def attack_row(name, y_true, y_pred, y_score):
    return {
        "attack": name,
        "auc": roc_auc_score(y_true, y_score),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def mia_features_enriched(proba, y_true, eps=1e-6):
    p = np.clip(np.asarray(proba, dtype=np.float64), eps, 1.0 - eps)
    y_true = np.asarray(y_true, dtype=np.int32)

    loss = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    conf = np.maximum(p, 1 - p)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    cor = ((p >= 0.5).astype(np.int32) == y_true).astype(np.float64)
    margin = np.abs(p - 0.5)

    logit = np.log(p / (1 - p))
    p_true = np.where(y_true == 1, p, 1 - p)
    p_false = 1.0 - p_true

    feats = np.column_stack([loss, conf, ent, cor, margin, p, logit, p_true, p_false])
    return np.nan_to_num(feats, nan=0.0, posinf=1e6, neginf=-1e6)


def tpr_at_fpr(y_true, y_score, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    idx = np.searchsorted(fpr, target_fpr, side='right') - 1
    idx = np.clip(idx, 0, len(tpr) - 1)
    return float(tpr[idx])


def rank01(x):
    x = np.asarray(x)
    order = np.argsort(np.argsort(x))
    if len(x) <= 1:
        return np.zeros_like(x, dtype=np.float64)
    return order.astype(np.float64) / float(len(x) - 1)


def normal_logpdf(x, mu, sigma):
    sigma = max(float(sigma), 1e-3)
    z = (x - mu) / sigma
    return -0.5 * (np.log(2.0 * np.pi) + 2.0 * np.log(sigma) + z * z)


def predict_label_blackbox(model, X_2d):
    X_seq = X_2d.reshape(-1, 1, X_2d.shape[1])
    p = model.predict(X_seq, verbose=0).ravel()
    return (p >= 0.5).astype(np.int32)


def boundary_distance_label_only(model, x, pred_label, seed, max_alpha=2.0, n_dirs=24, n_steps=8):
    rng = np.random.default_rng(seed)
    d = rng.normal(size=(n_dirs, x.shape[0]))
    d = d / (np.linalg.norm(d, axis=1, keepdims=True) + 1e-12)

    alphas = np.geomspace(0.02, max_alpha, num=n_steps)
    candidates = []
    for i in range(n_dirs):
        for a in alphas:
            candidates.append(x + a * d[i])
    candidates = np.asarray(candidates, dtype=np.float32)

    labels = predict_label_blackbox(model, candidates)
    labels = labels.reshape(n_dirs, n_steps)
    flips = labels != int(pred_label)

    dist = max_alpha * 1.5
    for i in range(n_dirs):
        where = np.where(flips[i])[0]
        if len(where) > 0:
            cand = float(alphas[int(where[0])])
            if cand < dist:
                dist = cand
    return dist

In [3]:
print("=== Chargement des donnees preparees ===")
prepared_path = os.path.join("..", "data_preparation", "prepared_oasis2_transformer.npz")
bundle = np.load(prepared_path, allow_pickle=True)

X = bundle["X"]
y = bundle["y"]
X_train = bundle["X_train"]
X_test = bundle["X_test"]
X_train_seq = bundle["X_train_seq"]
X_test_seq = bundle["X_test_seq"]
X_shadow_raw = bundle["X_shadow_raw"]
y_shadow = bundle["y_shadow"]
y_train = bundle["y_train"]
y_test = bundle["y_test"]
FEATURE_COLS = bundle["feature_cols"].tolist()
SEED = int(bundle["seed"][0])

print(f"Fichier charge: {prepared_path}")
print(f"X shape: {X.shape} | y shape: {y.shape}")
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Shadow pool size: {len(X_shadow_raw)}")
print(f"Train ratio classe 1: {y_train.mean():.4f} | Test ratio classe 1: {y_test.mean():.4f}")

=== Chargement des donnees preparees ===
Fichier charge: ..\data_preparation\prepared_oasis2_transformer.npz
X shape: (354, 9) | y shape: (354,)
Train size: 70 | Test size: 284
Shadow pool size: 107
Train ratio classe 1: 0.3571 | Test ratio classe 1: 0.3592


In [4]:
print("=== Entrainement Transformer baseline ===")

tf.keras.backend.clear_session()
set_global_determinism(SEED + 10)
target_model = build_transformer(n_features=X_train.shape[1], dropout=0.15)

es = EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
    verbose=0,
)

history = target_model.fit(
    X_train_seq,
    y_train,
    epochs=120,
    batch_size=32,
    validation_split=0.2,
    callbacks=[es],
    verbose=0,
)

p_train = target_model.predict(X_train_seq, verbose=0).ravel()
p_test = target_model.predict(X_test_seq, verbose=0).ravel()

train_acc = accuracy_score(y_train, (p_train >= 0.5).astype(int))
test_acc = accuracy_score(y_test, (p_test >= 0.5).astype(int))
train_auc = roc_auc_score(y_train, p_train)
test_auc = roc_auc_score(y_test, p_test)

best_val_acc = max(history.history["val_accuracy"]) if "val_accuracy" in history.history else float("nan")
best_val_loss = min(history.history["val_loss"]) if "val_loss" in history.history else float("nan")

transformer_metrics = pd.DataFrame(
    [{
        "train_acc": train_acc,
        "test_acc": test_acc,
        "gap": train_acc - test_acc,
        "train_auc": train_auc,
        "test_auc": test_auc,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "epochs_ran": len(history.history["loss"]),
    }]
)

print(f"Derniere loss train: {history.history['loss'][-1]:.6f}")
print(f"Meilleure val_acc: {best_val_acc:.4f} | Meilleure val_loss: {best_val_loss:.6f}")
display(transformer_metrics.round(4))

print("\n=== Modele cible MIA (stress-test) ===")
tf.keras.backend.clear_session()
set_global_determinism(SEED + 20)
mia_target_model = build_transformer(n_features=X_train.shape[1], dropout=0.0)
mia_target_model.fit(
    X_train_seq,
    y_train,
    epochs=260,
    batch_size=32,
    verbose=0,
)

p_train_mia = mia_target_model.predict(X_train_seq, verbose=0).ravel()
p_test_mia = mia_target_model.predict(X_test_seq, verbose=0).ravel()

print(f"train_acc_mia={accuracy_score(y_train, (p_train_mia >= 0.5).astype(int)):.4f}")
print(f"test_acc_mia={accuracy_score(y_test, (p_test_mia >= 0.5).astype(int)):.4f}")

=== Entrainement Transformer baseline ===
Derniere loss train: 0.029273
Meilleure val_acc: 1.0000 | Meilleure val_loss: 0.204655


,train_acc,test_acc,gap,train_auc,test_auc,best_val_acc,best_val_loss,epochs_ran
0,0.9714,0.9331,0.0383,0.9956,0.9892,1.0,0.2047,18



=== Modele cible MIA (stress-test) ===
train_acc_mia=1.0000
test_acc_mia=0.9331


In [5]:
print("=== Features MIA et stats ===")
F_mem = mia_features(p_train_mia, y_train)
F_non = mia_features(p_test_mia, y_test)
X_attack = np.vstack([F_mem, F_non])
y_attack = np.concatenate([
    np.ones(len(F_mem), dtype=int),
    np.zeros(len(F_non), dtype=int),
])

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_attack, y_attack, test_size=0.4, random_state=SEED, stratify=y_attack
)

print(f"F_mem: {F_mem.shape} | F_non: {F_non.shape}")
print(f"X_attack: {X_attack.shape}")
print(f"Attack train: {Xa_tr.shape} | Attack test: {Xa_te.shape}")
print(f"Loss mean members: {F_mem[:,0].mean():.6f}")
print(f"Loss mean non-members: {F_non[:,0].mean():.6f}")

=== Features MIA et stats ===
F_mem: (70, 5) | F_non: (284, 5)
X_attack: (354, 5)
Attack train: (212, 5) | Attack test: (142, 5)
Loss mean members: 0.000092
Loss mean non-members: 0.367018


In [6]:
print("=== Attaque 1: Threshold Loss ===")
score_tr = -Xa_tr[:, 0]
score_te = -Xa_te[:, 0]

cand = np.linspace(score_tr.min(), score_tr.max(), 400)
best_thr = cand[0]
best_bal = -1.0

for t in cand:
    pred = (score_tr >= t).astype(int)
    tpr = ((pred == 1) & (ya_tr == 1)).sum() / max((ya_tr == 1).sum(), 1)
    tnr = ((pred == 0) & (ya_tr == 0)).sum() / max((ya_tr == 0).sum(), 1)
    bal = 0.5 * (tpr + tnr)
    if bal > best_bal:
        best_bal = bal
        best_thr = t

pred_thr = (score_te >= best_thr).astype(int)
thr_metrics = attack_row("threshold_loss", ya_te, pred_thr, score_te)

print(f"Best threshold: {best_thr:.6f}")
print(f"Balanced accuracy (train attack): {best_bal:.4f}")
display(pd.DataFrame([thr_metrics]).round(4))

=== Attaque 1: Threshold Loss ===
Best threshold: -0.023456
Balanced accuracy (train attack): 0.5353


,attack,auc,accuracy,precision,recall,f1
0,threshold_loss,0.4821,0.3099,0.2222,1.0,0.3636


In [7]:
print("=== Attaque 2: Logistic Regression ===")
lr_attack = LogisticRegression(max_iter=1000, random_state=SEED)
lr_attack.fit(Xa_tr, ya_tr)
proba_lr = lr_attack.predict_proba(Xa_te)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)

lr_metrics = attack_row("logistic", ya_te, pred_lr, proba_lr)

print(f"Coefficients: {lr_attack.coef_.shape}")
display(pd.DataFrame([lr_metrics]).round(4))

=== Attaque 2: Logistic Regression ===
Coefficients: (1, 5)


,attack,auc,accuracy,precision,recall,f1
0,logistic,0.4818,0.8028,0.0,0.0,0.0


In [8]:
print("=== Attaque 3: Shadow Meta ===")

X_ab, X_aux, y_ab, y_aux = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_target_mem, X_target_non, y_target_mem, y_target_non = train_test_split(
    X_ab, y_ab, test_size=0.45, random_state=SEED + 1, stratify=y_ab
)

sc_t = StandardScaler()
X_target_mem_sc = sc_t.fit_transform(X_target_mem).astype(np.float32)
X_target_non_sc = sc_t.transform(X_target_non).astype(np.float32)
X_target_mem_seq = X_target_mem_sc.reshape(-1, 1, X_target_mem_sc.shape[1])
X_target_non_seq = X_target_non_sc.reshape(-1, 1, X_target_non_sc.shape[1])

tf.keras.backend.clear_session()
set_global_determinism(SEED + 1000)
target_shadow_eval_model = build_transformer(n_features=X_target_mem_sc.shape[1], dropout=0.0)
target_shadow_eval_model.fit(X_target_mem_seq, y_target_mem, epochs=220, batch_size=16, verbose=0)

p_mem_target = target_shadow_eval_model.predict(X_target_mem_seq, verbose=0).ravel()
p_non_target = target_shadow_eval_model.predict(X_target_non_seq, verbose=0).ravel()

F_mem_target = mia_features_enriched(p_mem_target, y_target_mem)
F_non_target = mia_features_enriched(p_non_target, y_target_non)
X_attack_target = np.vstack([F_mem_target, F_non_target])
y_attack_target = np.concatenate([
    np.ones(len(F_mem_target), dtype=int),
    np.zeros(len(F_non_target), dtype=int),
])
y_attack_target_class = np.concatenate([
    y_target_mem.astype(int),
    y_target_non.astype(int),
])

X_query_target = np.vstack([X_target_mem_sc, X_target_non_sc])
y_pred_target = np.concatenate([
    (p_mem_target >= 0.5).astype(np.int32),
    (p_non_target >= 0.5).astype(np.int32),
])

shadow_features = []
shadow_labels = []
shadow_cls = []
shadow_in_scores = {0: [], 1: []}
shadow_out_scores = {0: [], 1: []}

n_shadows = 50
for s in range(n_shadows):
    xs_mem, xs_non, ys_mem, ys_non = train_test_split(
        X_aux, y_aux, test_size=0.5, random_state=SEED + 100 + s, stratify=y_aux
    )

    sc_s = StandardScaler()
    xs_mem_sc = sc_s.fit_transform(xs_mem).astype(np.float32)
    xs_non_sc = sc_s.transform(xs_non).astype(np.float32)

    xs_mem_seq = xs_mem_sc.reshape(-1, 1, xs_mem_sc.shape[1])
    xs_non_seq = xs_non_sc.reshape(-1, 1, xs_non_sc.shape[1])

    tf.keras.backend.clear_session()
    set_global_determinism(SEED + 2000 + s)
    sh_model = build_transformer(n_features=xs_mem_sc.shape[1], dropout=0.0)
    sh_model.fit(xs_mem_seq, ys_mem, epochs=140, batch_size=16, verbose=0)

    ps_mem = sh_model.predict(xs_mem_seq, verbose=0).ravel()
    ps_non = sh_model.predict(xs_non_seq, verbose=0).ravel()

    fm = mia_features_enriched(ps_mem, ys_mem)
    fn = mia_features_enriched(ps_non, ys_non)
    shadow_features.append(np.vstack([fm, fn]))
    shadow_labels.append(np.concatenate([
        np.ones(len(fm), dtype=int),
        np.zeros(len(fn), dtype=int),
    ]))
    shadow_cls.append(np.concatenate([
        ys_mem.astype(int),
        ys_non.astype(int),
    ]))

    p_true_mem = np.where(ys_mem == 1, ps_mem, 1.0 - ps_mem)
    p_true_non = np.where(ys_non == 1, ps_non, 1.0 - ps_non)
    s_mem = np.log(np.clip(p_true_mem, 1e-6, 1 - 1e-6) / np.clip(1.0 - p_true_mem, 1e-6, 1 - 1e-6))
    s_non = np.log(np.clip(p_true_non, 1e-6, 1 - 1e-6) / np.clip(1.0 - p_true_non, 1e-6, 1 - 1e-6))

    for cls in [0, 1]:
        shadow_in_scores[cls].extend(s_mem[ys_mem == cls].tolist())
        shadow_out_scores[cls].extend(s_non[ys_non == cls].tolist())

X_shadow = np.vstack(shadow_features)
y_shadow_m = np.concatenate(shadow_labels)
y_shadow_cls = np.concatenate(shadow_cls)

lira_params = {}
for cls in [0, 1]:
    arr_in = np.asarray(shadow_in_scores[cls], dtype=np.float64)
    arr_out = np.asarray(shadow_out_scores[cls], dtype=np.float64)
    lira_params[cls] = {
        'mu_in': float(arr_in.mean()) if len(arr_in) else 0.0,
        'sd_in': float(arr_in.std(ddof=1)) if len(arr_in) > 1 else 1.0,
        'mu_out': float(arr_out.mean()) if len(arr_out) else 0.0,
        'sd_out': float(arr_out.std(ddof=1)) if len(arr_out) > 1 else 1.0,
    }

boundary_raw = np.zeros(len(X_query_target), dtype=np.float64)
for i in range(len(X_query_target)):
    boundary_raw[i] = boundary_distance_label_only(
        target_shadow_eval_model,
        X_query_target[i],
        y_pred_target[i],
        seed=SEED + 9000 + i,
        max_alpha=2.0,
        n_dirs=24,
        n_steps=8,
    )
score_boundary_all = rank01(boundary_raw)

seed_list = [11, 22, 33, 44, 55]
rows_seed = []

for s_eval in seed_list:
    Xa_tr_full, Xa_te_s, ya_tr_full, ya_te_s, yc_tr_full, yc_te_s, sb_tr_full, sb_te_s = train_test_split(
        X_attack_target,
        y_attack_target,
        y_attack_target_class,
        score_boundary_all,
        test_size=0.4,
        random_state=s_eval,
        stratify=y_attack_target,
    )

    Xa_tr_s, Xa_val, ya_tr_s, ya_val, yc_tr_s, yc_val, sb_tr_s, sb_val = train_test_split(
        Xa_tr_full,
        ya_tr_full,
        yc_tr_full,
        sb_tr_full,
        test_size=0.25,
        random_state=s_eval + 99,
        stratify=ya_tr_full,
    )

    X_meta_train = np.vstack([X_shadow, Xa_tr_s])
    y_meta_train = np.concatenate([y_shadow_m, ya_tr_s])
    yc_meta_train = np.concatenate([y_shadow_cls, yc_tr_s])

    models_by_cls = {}
    for cls in [0, 1]:
        msk = yc_meta_train == cls
        if msk.sum() >= 20 and len(np.unique(y_meta_train[msk])) == 2:
            m_cls = GradientBoostingClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=3,
                random_state=SEED + cls + s_eval,
            )
            m_cls.fit(X_meta_train[msk], y_meta_train[msk])
            models_by_cls[cls] = m_cls

    global_meta = GradientBoostingClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED + s_eval,
    )
    global_meta.fit(X_meta_train, y_meta_train)

    def score_meta_fn(Xa, yc):
        out = np.zeros(len(Xa), dtype=np.float64)
        for i in range(len(Xa)):
            cls = int(yc[i])
            model_i = models_by_cls.get(cls, global_meta)
            out[i] = model_i.predict_proba(Xa[i:i+1])[:, 1][0]
        return out

    def score_lira_fn(Xa, yc):
        p_true = np.clip(Xa[:, 7], 1e-6, 1 - 1e-6)
        s_vals = np.log(p_true / (1.0 - p_true))
        out = np.zeros(len(Xa), dtype=np.float64)
        for i in range(len(Xa)):
            prm = lira_params[int(yc[i])]
            li = normal_logpdf(s_vals[i], prm['mu_in'], prm['sd_in'])
            lo = normal_logpdf(s_vals[i], prm['mu_out'], prm['sd_out'])
            out[i] = li - lo
        return out

    score_meta_val = score_meta_fn(Xa_val, yc_val)
    score_lira_val = score_lira_fn(Xa_val, yc_val)
    score_meta_te = score_meta_fn(Xa_te_s, yc_te_s)
    score_lira_te = score_lira_fn(Xa_te_s, yc_te_s)

    candidates = {
        'meta_only': rank01(score_meta_val),
        'lira_only': rank01(score_lira_val),
        'boundary_only': rank01(sb_val),
        'fusion_0.45_0.25_0.30': 0.45 * rank01(score_meta_val) + 0.25 * rank01(score_lira_val) + 0.30 * rank01(sb_val),
        'fusion_0.35_0.45_0.20': 0.35 * rank01(score_meta_val) + 0.45 * rank01(score_lira_val) + 0.20 * rank01(sb_val),
        'fusion_0.30_0.30_0.40': 0.30 * rank01(score_meta_val) + 0.30 * rank01(score_lira_val) + 0.40 * rank01(sb_val),
        'fusion_0.40_0.35_0.25': 0.40 * rank01(score_meta_val) + 0.35 * rank01(score_lira_val) + 0.25 * rank01(sb_val),
        'fusion_0.50_0.30_0.20': 0.50 * rank01(score_meta_val) + 0.30 * rank01(score_lira_val) + 0.20 * rank01(sb_val),
        'fusion_0.25_0.50_0.25': 0.25 * rank01(score_meta_val) + 0.50 * rank01(score_lira_val) + 0.25 * rank01(sb_val),
    }

    best_name = None
    best_auc_val = -1.0
    for name, sc in candidates.items():
        a = roc_auc_score(ya_val, sc)
        if a > best_auc_val:
            best_auc_val = a
            best_name = name

    if best_name == 'meta_only':
        score_fusion = rank01(score_meta_te)
    elif best_name == 'lira_only':
        score_fusion = rank01(score_lira_te)
    elif best_name == 'boundary_only':
        score_fusion = rank01(sb_te_s)
    elif best_name == 'fusion_0.45_0.25_0.30':
        score_fusion = 0.45 * rank01(score_meta_te) + 0.25 * rank01(score_lira_te) + 0.30 * rank01(sb_te_s)
    elif best_name == 'fusion_0.35_0.45_0.20':
        score_fusion = 0.35 * rank01(score_meta_te) + 0.45 * rank01(score_lira_te) + 0.20 * rank01(sb_te_s)
    else:
        score_fusion = 0.30 * rank01(score_meta_te) + 0.30 * rank01(score_lira_te) + 0.40 * rank01(sb_te_s)

    pred_shadow = (score_fusion >= 0.5).astype(int)

    row = attack_row('shadow_meta', ya_te_s, pred_shadow, score_fusion)
    row['seed'] = s_eval
    row['selected_scorer'] = best_name
    row['selected_val_auc'] = float(best_auc_val)
    row['auc_meta_only'] = roc_auc_score(ya_te_s, score_meta_te)
    row['auc_lira_only'] = roc_auc_score(ya_te_s, score_lira_te)
    row['auc_boundary_only'] = roc_auc_score(ya_te_s, sb_te_s)
    row['tpr_at_1pct_fpr'] = tpr_at_fpr(ya_te_s, score_fusion, 0.01)
    row['tpr_at_5pct_fpr'] = tpr_at_fpr(ya_te_s, score_fusion, 0.05)
    rows_seed.append(row)

shadow_seed_df = pd.DataFrame(rows_seed)

shadow_metrics = {
    'attack': 'shadow_meta',
    'auc': float(shadow_seed_df['auc'].mean()),
    'accuracy': float(shadow_seed_df['accuracy'].mean()),
    'precision': float(shadow_seed_df['precision'].mean()),
    'recall': float(shadow_seed_df['recall'].mean()),
    'f1': float(shadow_seed_df['f1'].mean()),
}

print(f"Split sizes | target_mem: {X_target_mem.shape} | target_non: {X_target_non.shape} | aux: {X_aux.shape}")
print(f"X_shadow: {X_shadow.shape} | n_shadows: {n_shadows} | features: {X_shadow.shape[1]}")
print(f"AUC meta-only mean: {shadow_seed_df['auc_meta_only'].mean():.4f}")
print(f"AUC lira-only mean: {shadow_seed_df['auc_lira_only'].mean():.4f}")
print(f"AUC boundary-only mean: {shadow_seed_df['auc_boundary_only'].mean():.4f}")
display(pd.DataFrame([shadow_metrics]).round(4))

=== Attaque 3: Shadow Meta ===
Split sizes | target_mem: (135, 9) | target_non: (112, 9) | aux: (107, 9)
X_shadow: (5350, 9) | n_shadows: 50 | features: 9
AUC meta-only mean: 0.5732
AUC lira-only mean: 0.5802
AUC boundary-only mean: 0.6281


,attack,auc,accuracy,precision,recall,f1
0,shadow_meta,0.6151,0.6485,0.6807,0.6815,0.6802


In [9]:
print("=== Tableau recapitulatif des AUC ===")
attack_metrics = pd.DataFrame([
    thr_metrics,
    lr_metrics,
    shadow_metrics,
]).sort_values("auc", ascending=False)

display(attack_metrics.round(4))

best_attack = attack_metrics.iloc[0]
print(f"Best MIA: {best_attack['attack']} | AUC={best_attack['auc']:.4f}")

=== Tableau recapitulatif des AUC ===


,attack,auc,accuracy,precision,recall,f1
2,shadow_meta,0.6151,0.6485,0.6807,0.6815,0.6802
0,threshold_loss,0.4821,0.3099,0.2222,1.0000,0.3636
1,logistic,0.4818,0.8028,0.0000,0.0000,0.0000


Best MIA: shadow_meta | AUC=0.6151
